# 03 Infer — 加载 LoRA + Gradio 多轮对话

TaleTalk 流水线第三阶段：加载 `02_train.ipynb` 训练好的 LoRA，merge 进 base model，启动 Gradio Chatbot 多轮聊天界面，配合 Cloudflare 隧道出公网。

**多轮上下文**：history 会真正拼进 prompt，所以模型能记住前面聊过什么。
**流式输出**：边生成边吐字，不用干等 4-6 秒。
**复读防护**：`repetition_penalty=1.1` + `no_repeat_ngram_size=4` + 多种 stop pattern。

## 0. 参数区

`RUN_NAME` 和 `MODEL_CHOICE` 必须和 02_train 一致，否则找不到训练好的 checkpoint。

In [ ]:
from pathlib import Path
import os, sys, json, re, subprocess, time

RUN_NAME = 'shiri_qixia'        # 和 02_train 保持一致
MODEL_CHOICE = 'qwen3_5_9b'     # 和 02_train 保持一致
TARGET_ROLE = '齐夏'
NOVEL_TITLE = '十日终焉'
RUN_TAG = ''                    # 和 02_train 保持一致

MODEL_IDS = {
    'qwen3_5_9b':       'Qwen/Qwen3.5-9B',
    'qwen3_5_27b':      'Qwen/Qwen3.5-27B',
    'qwen3_6_35b_a3b':  'Qwen/Qwen3.6-35B-A3B',
    'qwen3_8b':         'Qwen/Qwen3-8B',
}
MODEL_ID = MODEL_IDS[MODEL_CHOICE]

# ====== 路径 ======
REPO_DIR = Path.cwd().resolve()
while REPO_DIR != REPO_DIR.parent and not (REPO_DIR / 'extract').is_dir():
    REPO_DIR = REPO_DIR.parent
assert (REPO_DIR / 'extract').is_dir(), 'taletalk 仓库根目录找不到'

PERSIST_DIR = Path('/network-workspace')
MODEL_CACHE_DIR = PERSIST_DIR / 'models'
FULL_RUN_NAME = f"{RUN_NAME}_{MODEL_CHOICE}_lora" + (f"_{RUN_TAG}" if RUN_TAG else "")
OUTPUT_DIR = PERSIST_DIR / "outputs" / FULL_RUN_NAME

GRADIO_PORT = 7860

print('repo:', REPO_DIR)
print('full run name:', FULL_RUN_NAME)
print('output dir:', OUTPUT_DIR, 'exists=', OUTPUT_DIR.exists())

## 1. 加载 base + LoRA 并 merge

merge 后 LoRA adapter forward 那一层就没了，推理快 10-20%。

In [ ]:
import torch
from peft import PeftModel
from transformers import AutoTokenizer

# Qwen3.5 走 text-only loader（绕过 vision tower 在 torch 2.9 ROCm 的 Conv3D bug）
try:
    from transformers import Qwen3_5ForCausalLM as BaseLM
except ImportError:
    from transformers import AutoModelForCausalLM as BaseLM

def checkpoint_step(p):
    m = re.search(r'checkpoint-(\d+)$', p.name)
    return int(m.group(1)) if m else -1

checkpoints = [p for p in OUTPUT_DIR.glob('checkpoint-*') if (p / 'adapter_model.safetensors').exists()]
ADAPTER_DIR = max(checkpoints, key=checkpoint_step) if checkpoints else OUTPUT_DIR
assert (ADAPTER_DIR / 'adapter_model.safetensors').exists(), f'未找到 LoRA: {ADAPTER_DIR}'

# base model 路径优先用 modelscope 缓存
ms_dir = MODEL_CACHE_DIR / 'Qwen' / MODEL_ID.split('/')[-1].replace('.', '___')
BASE_MODEL = ms_dir if ms_dir.exists() else MODEL_ID

print('base_model:', BASE_MODEL)
print('adapter:', ADAPTER_DIR)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
base = BaseLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16,
    device_map={'': 0},
    trust_remote_code=True,
    attn_implementation='sdpa',
)
print('merging LoRA into base ...')
model = PeftModel.from_pretrained(base, ADAPTER_DIR)
model = model.merge_and_unload()
model.eval()
print('ready. param dtype:', next(model.parameters()).dtype, 'device:', next(model.parameters()).device)

## 2. 多轮聊天函数（流式）

history 真正拼进 prompt，模型才会记住前几轮聊过什么。

In [ ]:
from threading import Thread
from transformers import TextIteratorStreamer

SYSTEM_PROMPT = (
    f'你正在扮演《{NOVEL_TITLE}》中的{TARGET_ROLE}。'
    f'严格保持{TARGET_ROLE}的语气、性格、说话习惯和价值观，'
    f'根据对话上下文自然回应，不要跳出角色，不要续写其他角色的发言。'
)

def build_prompt(message, history):
    parts = [SYSTEM_PROMPT, '']
    for h in history or []:
        if isinstance(h, (list, tuple)) and len(h) == 2:
            u, a = h
            if u:
                parts.append(f'用户：{u}')
            if a:
                parts.append(f'{TARGET_ROLE}：{a}')
    parts.append(f'用户：{message}')
    parts.append(f'{TARGET_ROLE}：')
    return '\n'.join(parts)

STOP_PATTERNS = [
    re.compile(r'(?i)(?:^|[\s\n])user\b'),
    re.compile(r'(?i)(?:^|[\s\n])assistant\b'),
    re.compile(r'用户\s*[:：]?'),
    re.compile(rf'{re.escape(TARGET_ROLE)}\s*[:：]?'),
    re.compile(r'<\|im_(?:start|end)\|>'),
    re.compile(r'\n{3,}'),
]

def chat_fn(message, history, max_new_tokens, temperature):
    prompt = build_prompt(message, history)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
    gen_kwargs = dict(
        **inputs,
        streamer=streamer,
        max_new_tokens=int(max_new_tokens),
        do_sample=float(temperature) > 0,
        top_p=0.9,
        repetition_penalty=1.1,
        no_repeat_ngram_size=4,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
    )
    if float(temperature) > 0:
        gen_kwargs['temperature'] = float(temperature)
    Thread(target=model.generate, kwargs=gen_kwargs).start()

    acc = ''
    for chunk in streamer:
        acc += chunk
        cut = len(acc)
        for pat in STOP_PATTERNS:
            m = pat.search(acc)
            if m and m.start() < cut:
                cut = m.start()
        if cut < len(acc):
            yield acc[:cut].rstrip(' \n\t：:，,。.').strip()
            return
        yield acc.rstrip()

## 3. Gradio UI + Cloudflare 隧道

- Gradio 用 `ChatInterface` + 流式 generator。
- 隧道用 cloudflared quick tunnel（72 小时有效），云端 gradio.live 通常不通。
- cloudflared 二进制走 gh-proxy 镜像下载，HF/GitHub 直连不稳的环境也能拉到。

In [ ]:
try:
    import gradio as gr
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'gradio'])
    import gradio as gr

EXAMPLES = [
    [f"你是谁？", 200, 0.5],
    [f"如果一个规则看起来互相矛盾，你会怎么判断？", 200, 0.5],
    [f"我现在很慌，怎么办？", 200, 0.5],
]

demo = gr.ChatInterface(
    fn=chat_fn,
    title=f"{TARGET_ROLE} LoRA 多轮对话",
    description=f"base: {BASE_MODEL}  |  adapter: {ADAPTER_DIR}",
    additional_inputs=[
        gr.Slider(32, 1024, value=200, step=8, label='max_new_tokens'),
        gr.Slider(0.0, 1.5, value=0.5, step=0.05, label='temperature'),
    ],
    examples=EXAMPLES,
)

In [ ]:
# 准备 cloudflared
CLOUDFLARED_PATH = Path('/opt/tunnels/cloudflared')
TUNNEL_LOG = Path('/tmp/cloudflared_tunnel.log')

if not CLOUDFLARED_PATH.exists() or CLOUDFLARED_PATH.stat().st_size < 30_000_000:
    CLOUDFLARED_PATH.parent.mkdir(parents=True, exist_ok=True)
    print('downloading cloudflared ...')
    subprocess.check_call([
        'curl', '-L', '--max-time', '180', '-o', str(CLOUDFLARED_PATH),
        'https://gh-proxy.org/https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
    ])
    CLOUDFLARED_PATH.chmod(0o755)
    print('cloudflared ready:', CLOUDFLARED_PATH)

# 杀掉同端口旧隧道
subprocess.run(['pkill', '-f', f'cloudflared.*:{GRADIO_PORT}'], capture_output=True)
time.sleep(0.5)

proc = subprocess.Popen(
    [str(CLOUDFLARED_PATH), 'tunnel', '--url', f'http://localhost:{GRADIO_PORT}'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    preexec_fn=os.setsid,
)
public_url = None
deadline = time.time() + 25
print('waiting for tunnel URL ...')
while time.time() < deadline:
    line = proc.stdout.readline()
    if not line: continue
    print(line.rstrip())
    m = re.search(r'https://[\w\-]+\.trycloudflare\.com', line)
    if m:
        public_url = m.group(0); break

if public_url:
    print('\n' + '=' * 80)
    print('✅ 在你本地浏览器打开:', public_url)
    print('=' * 80 + '\n')
else:
    print('⚠️ 未解析到隧道 URL，看上面 cloudflared 输出')

TUNNEL_PROC = proc

In [ ]:
demo.queue().launch(
    server_name='0.0.0.0',
    server_port=GRADIO_PORT,
    share=False,
    inline=False,
)

## 4. 可选：合并 LoRA 并导出完整模型

如果想脱离 LoRA 直接发布合并版基座（vLLM/部署友好），跑这个。

In [ ]:
MERGE = False  # 改成 True 才会执行合并导出

if MERGE:
    merged_out = PERSIST_DIR / 'outputs' / f'{FULL_RUN_NAME}_merged'
    cmd = f'python {REPO_DIR}/scripts/merge_lora.py --model {BASE_MODEL} --adapter {ADAPTER_DIR} --out {merged_out}'
    print(cmd)
    subprocess.check_call(cmd, shell=True)
    print(f"merged model: {merged_out}")
else:
    print('skip merge')